# Module 7 — Clean ATCNet-3C Reproduction + Strict A/B Target Normalization

This notebook is deliberately conservative.

It reproduces the **known-good ATCNet setup** used in the earlier BCI-IV-2a LOSO experiment that reached 80.56% standard ensemble and 82.87% TTA on S01 in the recorded run. The source logs also show the same `(1728, 22, 640) -> (1383,345,216)` fold geometry and seeds `(42,123)`. fileciteturn23file3L200-L212

### Experiment A — strict reproduction
- Source = the other 8 BCI-IV-2a subjects
- 80/20 stratified trial validation
- source-only robust normalization
- ATCNet-3C
- seeds 42 and 123
- probability ensemble
- **no target-derived statistics**

### Experiment B — target-normalization A/B
Same trained source models, but the held-out target EEG is additionally normalized using its own **unlabeled** target statistics. Target labels are never used for fitting/selection.

**Important:** Experiment B is transductive/unsupervised target normalization and must be reported separately from strict LOSO.

In [1]:
# ============================================================
# CELL 1 — IMPORTS / DEVICE / REPRODUCIBILITY
# ============================================================

from __future__ import annotations

import os
import gc
import copy
import time
import random
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import h5py

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import (
    TensorDataset,
    DataLoader,
    WeightedRandomSampler,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

SEED = 42


def seed_everything(seed=SEED):

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


seed_everything(SEED)

device = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

print("Device:", device)
print("Seed:", SEED)

Device: cpu
Seed: 42


In [2]:
# ============================================================
# CELL 2 — EXISTING PROJECT CACHE
# ============================================================

PROJECT_ROOT = Path(
    "/Users/ashokvarmabevara/Project2"
)

PROJECT_DIR = (
    PROJECT_ROOT
    / "cross_dataset_mi_project"
)

CACHE_DIR = (
    PROJECT_DIR
    / "cache"
)

RESULT_DIR = (
    PROJECT_DIR
    / "results"
    / "module_7_atcnet_reproduction"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CACHE_PATH = (
    CACHE_DIR
    / "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
)

if not CACHE_PATH.exists():

    candidates = sorted(
        CACHE_DIR.glob("*.h5")
    )

    preferred = [
        p
        for p in candidates
        if (
            "160hz" in p.name.lower()
            or "module_5" in p.name.lower()
        )
    ]

    if not preferred:
        raise FileNotFoundError(
            f"No HDF5 cache found in {CACHE_DIR}"
        )

    CACHE_PATH = preferred[0]


print(
    "Cache:",
    CACHE_PATH
)

assert CACHE_PATH.exists()

Cache: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5


In [3]:
# ============================================================
# CELL 3 — CACHE METADATA
# ============================================================

def decode(v):

    return (
        v.decode("utf-8")
        if isinstance(v, bytes)
        else str(v)
    )


with h5py.File(
    CACHE_PATH,
    "r",
) as h5:

    X_shape = tuple(
        h5["X"].shape
    )

    X_dtype = str(
        h5["X"].dtype
    )

    metadata = {}

    for key in [
        "dataset",
        "subject",
        "run",
        "recording_id",
        "filename",
        "absolute_path",
        "harmonized_class",
    ]:

        metadata[key] = [
            decode(v)
            for v in h5[
                "metadata"
            ][key][:]
        ]


cache_meta_df = pd.DataFrame(
    metadata
)

cache_meta_df.insert(
    0,
    "cache_index",
    np.arange(
        len(cache_meta_df),
        dtype=np.int64,
    ),
)

CLASSES = [
    "left",
    "right",
    "feet",
]

CLASS_TO_ID = {
    c: i
    for i, c in enumerate(CLASSES)
}

ID_TO_CLASS = {
    i: c
    for c, i in CLASS_TO_ID.items()
}

N_CLASSES = 3

assert X_shape[1:] == (
    22,
    640,
)

assert X_dtype == "float32"

bci_meta = (
    cache_meta_df[
        cache_meta_df[
            "dataset"
        ]
        .astype(str)
        == "BCI-IV-2a"
    ]
    .copy()
)

bci_meta["subject"] = (
    bci_meta[
        "subject"
    ].astype(str)
)

BCI_SUBJECTS = sorted(
    bci_meta[
        "subject"
    ].unique()
)

assert len(
    BCI_SUBJECTS
) == 9

print(
    "Cache:",
    X_shape
)

print(
    "BCI subjects:",
    BCI_SUBJECTS
)

print(
    "BCI epochs:",
    len(bci_meta)
)

Cache: (9316, 22, 640)
BCI subjects: ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09']
BCI epochs: 1944


In [4]:
# ============================================================
# CELL 4 — HDF5 LOADER + QA
# ============================================================

def load_indices(
    indices,
):

    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    with h5py.File(
        CACHE_PATH,
        "r",
    ) as h5:

        X = np.asarray(
            h5["X"][indices],
            dtype=np.float32,
        )

    return X


qa_indices = np.arange(
    min(
        512,
        X_shape[0],
    ),
    dtype=np.int64,
)

X_qa = load_indices(
    qa_indices
)

print(
    "Non-finite values:",
    int(
        (
            ~np.isfinite(
                X_qa
            )
        ).sum()
    )
)

print(
    "Zero-variance epochs:",
    int(
        (
            np.var(
                X_qa,
                axis=(1,2),
            )
            <= 1e-12
        ).sum()
    )
)

assert np.isfinite(
    X_qa
).all()

print(
    "✅ QA PASS"
)

Non-finite values: 0
Zero-variance epochs: 0
✅ QA PASS


In [5]:
# ============================================================
# CELL 5 — EXACT SOURCE-ONLY ROBUST NORMALIZER
# ============================================================

class SourceOnlyRobustNormalizer:

    def __init__(
        self,
        eps=1e-6,
    ):

        self.eps = eps
        self.median_ = None
        self.iqr_ = None

    def fit(
        self,
        X,
    ):

        X = np.asarray(
            X,
            dtype=np.float32,
        )

        values = (
            X
            .transpose(
                1,
                0,
                2,
            )
            .reshape(
                X.shape[1],
                -1,
            )
        )

        self.median_ = (
            np.median(
                values,
                axis=1,
            )
        )

        q25 = np.percentile(
            values,
            25,
            axis=1,
        )

        q75 = np.percentile(
            values,
            75,
            axis=1,
        )

        self.iqr_ = np.maximum(
            q75 - q25,
            self.eps,
        )

        return self

    def transform(
        self,
        X,
    ):

        if self.median_ is None:

            raise RuntimeError(
                "Normalizer has not been fitted."
            )

        X = np.asarray(
            X,
            dtype=np.float32,
        )

        Z = (
            X
            - self.median_[
                None,
                :,
                None,
            ]
        ) / (
            self.iqr_[
                None,
                :,
                None,
            ]
            + self.eps
        )

        Z = np.nan_to_num(
            Z,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        return Z.astype(
            np.float32
        )


def fit_target_specific_normalizer(
    X_target,
):

    target_norm = (
        SourceOnlyRobustNormalizer()
        .fit(
            X_target
        )
    )

    return (
        target_norm.transform(
            X_target
        ),
        target_norm,
    )

In [6]:
# ============================================================
# CELL 6 — STRATIFIED SOURCE TRIAL SPLIT
# ============================================================

def stratified_split(
    y,
    val_fraction=0.20,
    seed=SEED,
):

    y = np.asarray(
        y,
        dtype=np.int64,
    )

    rng = np.random.default_rng(
        seed
    )

    all_idx = np.arange(
        len(y),
        dtype=np.int64,
    )

    train_parts = []
    val_parts = []

    for cls in range(
        N_CLASSES
    ):

        idx = all_idx[
            y == cls
        ].copy()

        rng.shuffle(
            idx
        )

        n_val = max(
            1,
            int(
                round(
                    len(idx)
                    * val_fraction
                )
            ),
        )

        val_parts.append(
            idx[
                :n_val
            ]
        )

        train_parts.append(
            idx[
                n_val:
            ]
        )

    train_idx = np.concatenate(
        train_parts
    )

    val_idx = np.concatenate(
        val_parts
    )

    rng.shuffle(
        train_idx
    )

    rng.shuffle(
        val_idx
    )

    return (
        train_idx,
        val_idx,
    )


print(
    "✅ Split helper ready."
)

✅ Split helper ready.


In [7]:
# ============================================================
# CELL 7 — EXACT KNOWN-GOOD ATCNET TCN BLOCK
# ============================================================

class CausalConv1d(
    nn.Module
):

    def __init__(
        self,
        in_ch,
        out_ch,
        kernel_size,
        dilation=1,
        bias=False,
    ):

        super().__init__()

        self.pad = (
            kernel_size - 1
        ) * dilation

        self.conv = nn.Conv1d(
            in_ch,
            out_ch,
            kernel_size,
            padding=self.pad,
            dilation=dilation,
            bias=bias,
        )

    def forward(
        self,
        x,
    ):

        y = self.conv(
            x
        )

        if self.pad:

            y = y[
                ...,
                :-self.pad
            ]

        return y


class TCNResidualBlock(
    nn.Module
):

    def __init__(
        self,
        dim,
        filters=32,
        depth=2,
        kernel_size=4,
        dropout=0.30,
    ):

        super().__init__()

        self.proj = (
            nn.Conv1d(
                dim,
                filters,
                1,
            )
            if dim != filters
            else nn.Identity()
        )

        self.blocks = (
            nn.ModuleList()
        )

        for i in range(
            depth
        ):

            dilation = (
                2 ** i
            )

            self.blocks.append(
                nn.ModuleDict(
                    {
                        "c1":
                            CausalConv1d(
                                filters,
                                filters,
                                kernel_size,
                                dilation=dilation,
                            ),

                        "bn1":
                            nn.BatchNorm1d(
                                filters
                            ),

                        "c2":
                            CausalConv1d(
                                filters,
                                filters,
                                kernel_size,
                                dilation=dilation,
                            ),

                        "bn2":
                            nn.BatchNorm1d(
                                filters
                            ),

                        "drop":
                            nn.Dropout(
                                dropout
                            ),
                    }
                )
            )

    def forward(
        self,
        x,
    ):

        z = x.transpose(
            1,
            2,
        )

        residual = self.proj(
            z
        )

        for block in (
            self.blocks
        ):

            h = block[
                "c1"
            ](
                residual
            )

            h = block[
                "bn1"
            ](
                h
            )

            h = F.elu(
                h
            )

            h = block[
                "drop"
            ](
                h
            )

            h = block[
                "c2"
            ](
                h
            )

            h = block[
                "bn2"
            ](
                h
            )

            h = F.elu(
                h
            )

            h = block[
                "drop"
            ](
                h
            )

            residual = F.elu(
                residual + h
            )

        return residual.transpose(
            1,
            2,
        )

In [8]:
# ============================================================
# CELL 8 — EXACT KNOWN-GOOD ATCNET CNN FRONT END
# ============================================================

class ATCNetConvBlock(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        F1=16,
        D=2,
        kernel_size=64,
        pool1=8,
        pool2=7,
        dropout=0.30,
    ):

        super().__init__()

        F2 = F1 * D

        self.F2 = F2

        # IMPORTANT:
        # temporal convolution runs over TIME.
        self.temporal = nn.Conv2d(
            1,
            F1,
            kernel_size=(
                1,
                kernel_size,
            ),
            padding=(
                0,
                kernel_size // 2,
            ),
            bias=False,
        )

        self.bn1 = nn.BatchNorm2d(
            F1
        )

        # IMPORTANT:
        # depthwise spatial filtering across
        # the 22 EEG channels.
        self.spatial = nn.Conv2d(
            F1,
            F2,
            kernel_size=(
                n_channels,
                1,
            ),
            groups=F1,
            bias=False,
        )

        self.bn2 = nn.BatchNorm2d(
            F2
        )

        self.pool1 = nn.AvgPool2d(
            kernel_size=(
                1,
                pool1,
            ),
            stride=(
                1,
                pool1,
            ),
        )

        self.drop1 = nn.Dropout(
            dropout
        )

        self.refine = nn.Conv2d(
            F2,
            F2,
            kernel_size=(
                1,
                16,
            ),
            padding=(
                0,
                8,
            ),
            bias=False,
        )

        self.bn3 = nn.BatchNorm2d(
            F2
        )

        self.pool2 = nn.AvgPool2d(
            kernel_size=(
                1,
                pool2,
            ),
            stride=(
                1,
                pool2,
            ),
        )

        self.drop2 = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
    ):

        # B,C,T -> B,1,C,T
        z = x.unsqueeze(
            1
        )

        z = self.temporal(
            z
        )

        z = self.bn1(
            z
        )

        z = F.elu(
            z
        )

        z = self.spatial(
            z
        )

        z = self.bn2(
            z
        )

        z = F.elu(
            z
        )

        z = self.pool1(
            z
        )

        z = self.drop1(
            z
        )

        z = self.refine(
            z
        )

        z = self.bn3(
            z
        )

        z = F.elu(
            z
        )

        z = self.pool2(
            z
        )

        z = self.drop2(
            z
        )

        # B,F,1,T -> B,F,T
        z = z.squeeze(
            2
        )

        # B,F,T -> B,T,F
        z = z.transpose(
            1,
            2,
        )

        return z

In [9]:
# ============================================================
# CELL 9 — KNOWN-GOOD ATCNET-3C
# ============================================================

class ATCNet3C(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        n_samples=640,
        n_classes=3,
        n_windows=5,
        F1=16,
        D=2,
        attn_heads=2,
        attn_dropout=0.30,
        tcn_depth=2,
        tcn_kernel=4,
        tcn_filters=32,
        tcn_dropout=0.30,
        dropout=0.30,
    ):

        super().__init__()

        self.n_windows = (
            n_windows
        )

        self.n_classes = (
            n_classes
        )

        self.conv = (
            ATCNetConvBlock(
                n_channels=n_channels,
                F1=F1,
                D=D,
                kernel_size=kernel_size
                if "kernel_size" in locals()
                else 64,
                pool1=8,
                pool2=7,
                dropout=dropout,
            )
        )

        self.feature_dim = (
            F1 * D
        )

        self.attn = (
            nn.ModuleList(
                [
                    nn.MultiheadAttention(
                        embed_dim=self.feature_dim,
                        num_heads=attn_heads,
                        dropout=attn_dropout,
                        batch_first=True,
                    )
                    for _ in range(
                        n_windows
                    )
                ]
            )
        )

        self.attn_norm = (
            nn.ModuleList(
                [
                    nn.LayerNorm(
                        self.feature_dim
                    )
                    for _ in range(
                        n_windows
                    )
                ]
            )
        )

        self.tcn = (
            nn.ModuleList(
                [
                    TCNResidualBlock(
                        dim=self.feature_dim,
                        filters=tcn_filters,
                        depth=tcn_depth,
                        kernel_size=tcn_kernel,
                        dropout=tcn_dropout,
                    )
                    for _ in range(
                        n_windows
                    )
                ]
            )
        )

        self.window_head = (
            nn.ModuleList(
                [
                    nn.Sequential(
                        nn.Linear(
                            tcn_filters,
                            64,
                        ),

                        nn.ELU(),

                        nn.Dropout(
                            0.25
                        ),

                        nn.Linear(
                            64,
                            n_classes,
                        ),
                    )
                    for _ in range(
                        n_windows
                    )
                ]
            )
        )

        # Safely determine compressed sequence length.
        previous = self.training

        self.eval()

        with torch.no_grad():

            dummy = torch.zeros(
                2,
                n_channels,
                n_samples,
            )

            seq = self.conv(
                dummy
            )

        if previous:
            self.train()

        self.seq_len = int(
            seq.shape[1]
        )

        if self.seq_len < (
            n_windows
        ):

            raise RuntimeError(
                "Compressed sequence too short."
            )

    def forward(
        self,
        x,
    ):

        z = self.conv(
            x
        )

        logits = []

        for i in range(
            self.n_windows
        ):

            start = i

            end = (
                self.seq_len
                - self.n_windows
                + i
                + 1
            )

            w = z[
                :,
                start:end,
                :
            ]

            a, _ = (
                self.attn[i](
                    w,
                    w,
                    w,
                    need_weights=False,
                )
            )

            w = self.attn_norm[i](
                w + a
            )

            w = self.tcn[i](
                w
            )

            last = w[
                :,
                -1,
                :
            ]

            logits.append(
                self.window_head[i](
                    last
                )
            )

        return torch.stack(
            logits,
            dim=0,
        ).mean(
            dim=0
        )


# Forward test
test_model = ATCNet3C().to(
    device
)

test_model.eval()

with torch.no_grad():

    test_out = test_model(
        torch.randn(
            2,
            22,
            640,
            device=device,
        )
    )

print(
    "Compressed sequence:",
    test_model.seq_len
)

print(
    "Output:",
    tuple(
        test_out.shape
    )
)

print(
    "Parameters:",
    f"{sum(p.numel() for p in test_model.parameters() if p.requires_grad):,}"
)

assert tuple(
    test_out.shape
) == (
    2,
    3,
)

del test_model
gc.collect()

print(
    "✅ ATCNet-3C forward PASS"
)

Compressed sequence: 11
Output: (2, 3)
Parameters: 134,447
✅ ATCNet-3C forward PASS


In [10]:
# ============================================================
# CELL 10 — TRAINING AUGMENTATION + LOADERS
# ============================================================

def augment_training_eeg(
    x,
):

    x = x.clone()

    B, C, T = x.shape

    # Amplitude scaling
    if torch.rand(
        1,
        device=x.device,
    ).item() < 0.35:

        x = (
            x
            * torch.empty(
                B,
                1,
                1,
                device=x.device,
            ).uniform_(
                0.92,
                1.08,
            )
        )

    # Small sensor noise
    if torch.rand(
        1,
        device=x.device,
    ).item() < 0.20:

        x = (
            x
            + 0.004
            * torch.randn_like(
                x
            )
        )

    return x


def make_train_loader(
    X,
    y,
    batch_size=64,
):

    y = np.asarray(
        y,
        dtype=np.int64,
    )

    ds = TensorDataset(
        torch.from_numpy(
            np.asarray(
                X,
                dtype=np.float32,
            )
        ),
        torch.from_numpy(
            y
        ),
    )

    counts = np.bincount(
        y,
        minlength=N_CLASSES,
    ).astype(
        np.float64
    )

    inv = np.zeros(
        N_CLASSES,
        dtype=np.float64,
    )

    valid = counts > 0

    inv[valid] = (
        1.0
        / counts[valid]
    )

    weights = inv[y]

    sampler = WeightedRandomSampler(
        torch.as_tensor(
            weights,
            dtype=torch.double,
        ),
        num_samples=len(y),
        replacement=True,
    )

    return DataLoader(
        ds,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False,
        num_workers=0,
    )


def make_eval_loader(
    X,
    batch_size=128,
):

    ds = TensorDataset(
        torch.from_numpy(
            np.asarray(
                X,
                dtype=np.float32,
            )
        ),
        torch.zeros(
            len(X),
            dtype=torch.long,
        ),
    )

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

In [11]:
# ============================================================
# CELL 11 — PREDICTION + SOURCE TRAINING
# ============================================================

@torch.no_grad()
def predict_atcnet(
    model,
    X,
):

    model.eval()

    outputs = []

    for xb, _ in make_eval_loader(
        X
    ):

        xb = xb.to(
            device,
            non_blocking=True,
        )

        logits = model(
            xb
        )

        outputs.append(
            logits.detach()
            .cpu()
            .numpy()
        )

    logits = np.concatenate(
        outputs,
        axis=0,
    )

    logits -= logits.max(
        axis=1,
        keepdims=True,
    )

    P = np.exp(
        logits
    )

    P /= (
        P.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return P.astype(
        np.float32
    )


def train_source_atcnet(
    X_train,
    y_train,
    X_val,
    y_val,
    seed,
    epochs=150,
    batch_size=64,
    lr=9e-4,
    patience=30,
):

    seed_everything(
        seed
    )

    model = ATCNet3C().to(
        device
    )

    counts = np.bincount(
        y_train,
        minlength=N_CLASSES,
    ).astype(
        np.float32
    )

    class_weights = (
        counts.sum()
        /
        (
            N_CLASSES
            *
            np.maximum(
                counts,
                1.0,
            )
        )
    )

    class_weights /= (
        class_weights.mean()
        + 1e-12
    )

    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(
            class_weights,
            dtype=torch.float32,
            device=device,
        ),
        label_smoothing=0.01,
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
    )

    scheduler = (
        optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.90,
            patience=12,
            min_lr=1e-5,
        )
    )

    loader = make_train_loader(
        X_train,
        y_train,
        batch_size=batch_size,
    )

    best_state = None
    best_loss = np.inf
    best_bacc = -np.inf
    best_epoch = 0
    wait = 0

    history = []

    print(
        "\n"
        + "-" * 72
    )

    print(
        f"ATCNet seed = {seed}"
    )

    print(
        "-" * 72
    )

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        train_true = []
        train_pred = []
        train_losses = []

        for xb, yb in loader:

            xb = xb.to(
                device,
                non_blocking=True,
            )

            yb = yb.to(
                device,
                non_blocking=True,
            )

            xb = augment_training_eeg(
                xb
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                xb
            )

            loss = criterion(
                logits,
                yb
            )

            if not torch.isfinite(
                loss
            ):
                continue

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            train_losses.append(
                float(
                    loss.item()
                )
            )

            train_true.extend(
                yb.detach()
                .cpu()
                .numpy()
            )

            train_pred.extend(
                logits.argmax(
                    dim=1
                )
                .detach()
                .cpu()
                .numpy()
            )

        P_val = predict_atcnet(
            model,
            X_val,
        )

        pred_val = (
            P_val.argmax(
                axis=1
            )
        )

        val_loss = -float(
            np.mean(
                np.log(
                    np.clip(
                        P_val[
                            np.arange(
                                len(y_val)
                            ),
                            y_val,
                        ],
                        1e-8,
                        1.0,
                    )
                )
            )
        )

        train_acc = (
            accuracy_score(
                train_true,
                train_pred,
            )
            * 100.0
        )

        val_acc = (
            accuracy_score(
                y_val,
                pred_val,
            )
            * 100.0
        )

        val_bacc = (
            balanced_accuracy_score(
                y_val,
                pred_val,
            )
            * 100.0
        )

        scheduler.step(
            val_loss
        )

        history.append({
            "epoch":
                epoch,
            "train_acc":
                train_acc,
            "val_acc":
                val_acc,
            "val_bacc":
                val_bacc,
            "val_loss":
                val_loss,
            "lr":
                optimizer.param_groups[
                    0
                ]["lr"],
        })

        if val_loss < (
            best_loss
            - 1e-5
        ):

            best_loss = val_loss
            best_bacc = val_bacc
            best_epoch = epoch
            wait = 0

            best_state = copy.deepcopy(
                model.state_dict()
            )

        else:

            wait += 1

        if (
            epoch == 1
            or epoch % 10 == 0
        ):

            print(
                f"    epoch {epoch:03d} | "
                f"train={train_acc:5.1f}% | "
                f"val={val_acc:5.1f}% | "
                f"bAcc={val_bacc:5.1f}% | "
                f"vLoss={val_loss:.4f} | "
                f"lr={optimizer.param_groups[0]['lr']:.2e}"
            )

        if wait >= patience:

            print(
                f"    early stop at "
                f"epoch {epoch}; "
                f"best={best_epoch}"
            )

            break

    if best_state is None:

        raise RuntimeError(
            "No valid ATCNet checkpoint."
        )

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history),
        best_epoch,
        best_loss,
        best_bacc,
    )

In [12]:
# ============================================================
# CELL 12 — ONE S01 A/B FOLD
# ============================================================

def run_s01_ab():

    target_subject = "S01"

    source_mask = (
        bci_meta[
            "subject"
        ].astype(str)
        != target_subject
    )

    target_mask = (
        bci_meta[
            "subject"
        ].astype(str)
        == target_subject
    )

    source_idx = (
        bci_meta.loc[
            source_mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    target_idx = (
        bci_meta.loc[
            target_mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    X_source_raw = load_indices(
        source_idx
    )

    X_target_raw = load_indices(
        target_idx
    )

    y_source = (
        bci_meta.loc[
            source_mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    y_target = (
        bci_meta.loc[
            target_mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    # --------------------------------------------------------
    # SOURCE-ONLY NORMALIZATION
    # --------------------------------------------------------

    source_norm = (
        SourceOnlyRobustNormalizer()
        .fit(
            X_source_raw
        )
    )

    X_source = (
        source_norm.transform(
            X_source_raw
        )
    )

    # --------------------------------------------------------
    # A/B target preprocessing
    #
    # A:
    # source normalization applied to target
    #
    # B:
    # target-specific unlabeled normalization
    # --------------------------------------------------------

    X_target_A = (
        source_norm.transform(
            X_target_raw
        )
    )

    X_target_B, target_norm = (
        fit_target_specific_normalizer(
            X_target_raw
        )
    )

    # --------------------------------------------------------
    # Source validation
    # --------------------------------------------------------

    train_idx, val_idx = (
        stratified_split(
            y_source,
            val_fraction=0.20,
            seed=SEED,
        )
    )

    X_train = X_source[
        train_idx
    ]

    y_train = y_source[
        train_idx
    ]

    X_val = X_source[
        val_idx
    ]

    y_val = y_source[
        val_idx
    ]

    print(
        "\n"
        + "=" * 78
    )

    print(
        "ATCNet-3C S01 A/B TARGET NORMALIZATION"
    )

    print(
        "=" * 78
    )

    print(
        "Source:",
        X_source.shape,
    )

    print(
        "Train:",
        X_train.shape,
    )

    print(
        "Val:",
        X_val.shape,
    )

    print(
        "Target:",
        X_target_A.shape,
    )

    # --------------------------------------------------------
    # Two known-good seeds
    # --------------------------------------------------------

    models = []

    seed_summary = []

    for seed in (
        42,
        123,
    ):

        (
            model,
            history,
            best_epoch,
            best_loss,
            best_bacc,
        ) = train_source_atcnet(
            X_train,
            y_train,
            X_val,
            y_val,
            seed=seed,
            epochs=150,
            batch_size=64,
            lr=9e-4,
            patience=30,
        )

        models.append(
            model
        )

        seed_summary.append(
            {
                "seed":
                    seed,
                "best_epoch":
                    best_epoch,
                "best_val_loss":
                    best_loss,
                "best_val_bacc":
                    best_bacc,
            }
        )

    seed_df = pd.DataFrame(
        seed_summary
    )

    print(
        "\nSeed summary:"
    )

    display(
        seed_df
    )

    # --------------------------------------------------------
    # Validation ensemble
    # --------------------------------------------------------

    P_val = np.mean(
        np.stack(
            [
                predict_atcnet(
                    m,
                    X_val,
                )
                for m in models
            ],
            axis=0,
        ),
        axis=0,
    )

    val_pred = (
        P_val.argmax(
            axis=1
        )
    )

    val_acc = (
        accuracy_score(
            y_val,
            val_pred,
        )
        * 100.0
    )

    val_bacc = (
        balanced_accuracy_score(
            y_val,
            val_pred,
        )
        * 100.0
    )

    # --------------------------------------------------------
    # Experiment A: source-normalized target
    # --------------------------------------------------------

    P_A = np.mean(
        np.stack(
            [
                predict_atcnet(
                    m,
                    X_target_A,
                )
                for m in models
            ],
            axis=0,
        ),
        axis=0,
    )

    pred_A = (
        P_A.argmax(
            axis=1
        )
    )

    acc_A = (
        accuracy_score(
            y_target,
            pred_A,
        )
        * 100.0
    )

    bacc_A = (
        balanced_accuracy_score(
            y_target,
            pred_A,
        )
        * 100.0
    )

    kappa_A = (
        cohen_kappa_score(
            y_target,
            pred_A,
        )
    )

    # --------------------------------------------------------
    # Experiment B: target-specific unlabeled normalization
    # --------------------------------------------------------

    P_B = np.mean(
        np.stack(
            [
                predict_atcnet(
                    m,
                    X_target_B,
                )
                for m in models
            ],
            axis=0,
        ),
        axis=0,
    )

    pred_B = (
        P_B.argmax(
            axis=1
        )
    )

    acc_B = (
        accuracy_score(
            y_target,
            pred_B,
        )
        * 100.0
    )

    bacc_B = (
        balanced_accuracy_score(
            y_target,
            pred_B,
        )
        * 100.0
    )

    kappa_B = (
        cohen_kappa_score(
            y_target,
            pred_B,
        )
    )

    improvement = (
        acc_B
        - acc_A
    )

    print(
        "\n"
        + "-" * 78
    )

    print(
        f"Source validation bAcc : "
        f"{val_bacc:.2f}%"
    )

    print(
        f"A — strict target acc  : "
        f"{acc_A:.2f}%"
    )

    print(
        f"A — strict target bAcc : "
        f"{bacc_A:.2f}%"
    )

    print(
        f"A — kappa              : "
        f"{kappa_A:.4f}"
    )

    print(
        f"B — target-norm acc    : "
        f"{acc_B:.2f}%"
    )

    print(
        f"B — target-norm bAcc   : "
        f"{bacc_B:.2f}%"
    )

    print(
        f"B — kappa              : "
        f"{kappa_B:.4f}"
    )

    print(
        f"Normalization change   : "
        f"{improvement:+.2f} pp"
    )

    print(
        "-" * 78
    )

    return {
        "models":
            models,
        "seed_df":
            seed_df,
        "y_target":
            y_target,
        "pred_A":
            pred_A,
        "pred_B":
            pred_B,
        "P_A":
            P_A,
        "P_B":
            P_B,
        "val_bacc":
            val_bacc,
        "strict_acc":
            acc_A,
        "strict_bacc":
            bacc_A,
        "strict_kappa":
            kappa_A,
        "targetnorm_acc":
            acc_B,
        "targetnorm_bacc":
            bacc_B,
        "targetnorm_kappa":
            kappa_B,
        "improvement_pp":
            improvement,
    }


s01_ab = run_s01_ab()


ATCNet-3C S01 A/B TARGET NORMALIZATION
Source: (1728, 22, 640)
Train: (1383, 22, 640)
Val: (345, 22, 640)
Target: (216, 22, 640)

------------------------------------------------------------------------
ATCNet seed = 42
------------------------------------------------------------------------
    epoch 001 | train= 36.7% | val= 36.2% | bAcc= 36.2% | vLoss=1.0709 | lr=9.00e-04
    epoch 010 | train= 65.9% | val= 60.9% | bAcc= 60.9% | vLoss=0.8098 | lr=9.00e-04
    epoch 020 | train= 76.1% | val= 67.0% | bAcc= 67.0% | vLoss=0.7706 | lr=9.00e-04
    epoch 030 | train= 81.6% | val= 65.8% | bAcc= 65.8% | vLoss=0.7922 | lr=8.10e-04
    epoch 040 | train= 86.4% | val= 66.1% | bAcc= 66.1% | vLoss=0.9279 | lr=7.29e-04
    epoch 050 | train= 89.4% | val= 70.1% | bAcc= 70.1% | vLoss=0.8320 | lr=7.29e-04
    early stop at epoch 56; best=26

------------------------------------------------------------------------
ATCNet seed = 123
--------------------------------------------------------------------

,seed,best_epoch,best_val_loss,best_val_bacc
0,42,26,0.742525,66.956522
1,123,15,0.725572,68.695652



------------------------------------------------------------------------------
Source validation bAcc : 69.86%
A — strict target acc  : 70.37%
A — strict target bAcc : 70.37%
A — kappa              : 0.5556
B — target-norm acc    : 70.37%
B — target-norm bAcc   : 70.37%
B — kappa              : 0.5556
Normalization change   : +0.00 pp
------------------------------------------------------------------------------


In [14]:
# ============================================================
# CELL 13 — FULL 9-SUBJECT A/B LOSO
# ============================================================

RUN_FULL_LOSO = True

if RUN_FULL_LOSO:

    results = []
    fold_objects = {}

    for fold_id, target_subject in enumerate(
        BCI_SUBJECTS,
        start=1,
    ):

        target_subject = str(
            target_subject
        )

        # ----------------------------------------------------
        # Outer split
        # ----------------------------------------------------

        source_mask = (
            bci_meta[
                "subject"
            ].astype(str)
            != target_subject
        )

        target_mask = (
            bci_meta[
                "subject"
            ].astype(str)
            == target_subject
        )

        source_idx = (
            bci_meta.loc[
                source_mask,
                "cache_index",
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        target_idx = (
            bci_meta.loc[
                target_mask,
                "cache_index",
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        X_source_raw = load_indices(
            source_idx
        )

        X_target_raw = load_indices(
            target_idx
        )

        y_source = (
            bci_meta.loc[
                source_mask,
                "harmonized_class",
            ]
            .map(
                CLASS_TO_ID
            )
            .to_numpy(
                dtype=np.int64
            )
        )

        y_target = (
            bci_meta.loc[
                target_mask,
                "harmonized_class",
            ]
            .map(
                CLASS_TO_ID
            )
            .to_numpy(
                dtype=np.int64
            )
        )

        # Source normalizer.
        source_norm = (
            SourceOnlyRobustNormalizer()
            .fit(
                X_source_raw
            )
        )

        X_source = (
            source_norm.transform(
                X_source_raw
            )
        )

        # A: source normalization on target.
        X_target_A = (
            source_norm.transform(
                X_target_raw
            )
        )

        # B: target unlabeled statistics.
        X_target_B, _ = (
            fit_target_specific_normalizer(
                X_target_raw
            )
        )

        # Source validation.
        train_idx, val_idx = (
            stratified_split(
                y_source,
                val_fraction=0.20,
                seed=SEED,
            )
        )

        X_train = X_source[
            train_idx
        ]

        y_train = y_source[
            train_idx
        ]

        X_val = X_source[
            val_idx
        ]

        y_val = y_source[
            val_idx
        ]

        print(
            "\n"
            + "=" * 78
        )

        print(
            f"ATCNet A/B LOSO "
            f"[{fold_id}/9] — {target_subject}"
        )

        print(
            "=" * 78
        )

        print(
            "Train:",
            X_train.shape,
            "| Val:",
            X_val.shape,
            "| Target:",
            X_target_A.shape,
        )

        models = []

        for seed in (
            42,
            123,
        ):

            (
                model,
                _,
                _,
                _,
                _,
            ) = train_source_atcnet(
                X_train,
                y_train,
                X_val,
                y_val,
                seed=seed,
                epochs=150,
                batch_size=64,
                lr=9e-4,
                patience=30,
            )

            models.append(
                model
            )

        # A
        P_A = np.mean(
            np.stack(
                [
                    predict_atcnet(
                        m,
                        X_target_A,
                    )
                    for m in models
                ],
                axis=0,
            ),
            axis=0,
        )

        pred_A = P_A.argmax(
            axis=1
        )

        acc_A = (
            accuracy_score(
                y_target,
                pred_A,
            )
            * 100.0
        )

        bacc_A = (
            balanced_accuracy_score(
                y_target,
                pred_A,
            )
            * 100.0
        )

        # B
        P_B = np.mean(
            np.stack(
                [
                    predict_atcnet(
                        m,
                        X_target_B,
                    )
                    for m in models
                ],
                axis=0,
            ),
            axis=0,
        )

        pred_B = P_B.argmax(
            axis=1
        )

        acc_B = (
            accuracy_score(
                y_target,
                pred_B,
            )
            * 100.0
        )

        bacc_B = (
            balanced_accuracy_score(
                y_target,
                pred_B,
            )
            * 100.0
        )

        results.append({
            "fold":
                fold_id,
            "subject":
                target_subject,
            "strict_accuracy":
                acc_A,
            "strict_bacc":
                bacc_A,
            "targetnorm_accuracy":
                acc_B,
            "targetnorm_bacc":
                bacc_B,
            "delta_pp":
                acc_B - acc_A,
        })

        fold_objects[
            target_subject
        ] = {
            "y":
                y_target,
            "pred_A":
                pred_A,
            "pred_B":
                pred_B,
        }

        print(
            f"Strict A       : {acc_A:.2f}%"
        )

        print(
            f"Target-norm B  : {acc_B:.2f}%"
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(
        results
    )

    print(
        "\n"
        + "=" * 78
    )

    print(
        "FINAL A/B LOSO"
    )

    print(
        "=" * 78
    )

    display(
        results_df
    )

    print(
        "\nStrict mean:",
        f"{results_df['strict_accuracy'].mean():.2f}%"
    )

    print(
        "Target-normalized mean:",
        f"{results_df['targetnorm_accuracy'].mean():.2f}%"
    )

    print(
        "Strict mean bAcc:",
        f"{results_df['strict_bacc'].mean():.2f}%"
    )

    print(
        "Target-normalized mean bAcc:",
        f"{results_df['targetnorm_bacc'].mean():.2f}%"
    )

    print(
        "Mean improvement:",
        f"{results_df['delta_pp'].mean():+.2f} pp"
    )

    print(
        "Target-normalized subjects >=70:",
        int(
            (
                results_df[
                    "targetnorm_accuracy"
                ]
                >= 70
            ).sum()
        ),
        "/9",
    )

    print(
        "Target-normalized subjects >=80:",
        int(
            (
                results_df[
                    "targetnorm_accuracy"
                ]
                >= 80
            ).sum()
        ),
        "/9",
    )

    results_path = (
        RESULT_DIR
        / "atcnet_reproduction_ab_loso.csv"
    )

    results_df.to_csv(
        results_path,
        index=False,
    )

    print(
        "Saved:",
        results_path,
    )


ATCNet A/B LOSO [1/9] — S01
Train: (1383, 22, 640) | Val: (345, 22, 640) | Target: (216, 22, 640)

------------------------------------------------------------------------
ATCNet seed = 42
------------------------------------------------------------------------
    epoch 001 | train= 36.7% | val= 36.2% | bAcc= 36.2% | vLoss=1.0709 | lr=9.00e-04
    epoch 010 | train= 65.9% | val= 60.9% | bAcc= 60.9% | vLoss=0.8098 | lr=9.00e-04
    epoch 020 | train= 76.1% | val= 67.0% | bAcc= 67.0% | vLoss=0.7706 | lr=9.00e-04
    epoch 030 | train= 81.6% | val= 65.8% | bAcc= 65.8% | vLoss=0.7922 | lr=8.10e-04
    epoch 040 | train= 86.4% | val= 66.1% | bAcc= 66.1% | vLoss=0.9279 | lr=7.29e-04
    epoch 050 | train= 89.4% | val= 70.1% | bAcc= 70.1% | vLoss=0.8320 | lr=7.29e-04
    early stop at epoch 56; best=26

------------------------------------------------------------------------
ATCNet seed = 123
------------------------------------------------------------------------
    epoch 001 | train= 36.

,fold,subject,strict_accuracy,strict_bacc,targetnorm_accuracy,targetnorm_bacc,delta_pp
0,1,S01,70.370370,70.370370,70.370370,70.370370,0.000000
1,2,S02,37.962963,37.962963,43.981481,43.981481,6.018519
2,3,S03,69.444444,69.444444,72.685185,72.685185,3.240741
3,4,S04,40.277778,40.277778,43.518519,43.518519,3.240741
4,5,S05,37.962963,37.962963,43.981481,43.981481,6.018519
5,6,S06,38.888889,38.888889,55.555556,55.555556,16.666667
6,7,S07,39.814815,39.814815,38.888889,38.888889,-0.925926
7,8,S08,79.166667,79.166667,69.444444,69.444444,-9.722222
8,9,S09,71.296296,71.296296,63.425926,63.425926,-7.870370



Strict mean: 53.91%
Target-normalized mean: 55.76%
Strict mean bAcc: 53.91%
Target-normalized mean bAcc: 55.76%
Mean improvement: +1.85 pp
Target-normalized subjects >=70: 2 /9
Target-normalized subjects >=80: 0 /9
Saved: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/results/module_7_atcnet_reproduction/atcnet_reproduction_ab_loso.csv


In [ ]:
# ============================================================
# CELL 14 — FINAL CONFUSION MATRICES
# ============================================================

if (
    "fold_objects" in globals()
    and len(fold_objects) > 0
):

    y_all = np.concatenate([
        fold_objects[s]["y"]
        for s in BCI_SUBJECTS
    ])

    pred_A = np.concatenate([
        fold_objects[s]["pred_A"]
        for s in BCI_SUBJECTS
    ])

    pred_B = np.concatenate([
        fold_objects[s]["pred_B"]
        for s in BCI_SUBJECTS
    ])

    cm_A = confusion_matrix(
        y_all,
        pred_A,
        labels=[
            0,
            1,
            2,
        ],
        normalize="true",
    )

    cm_B = confusion_matrix(
        y_all,
        pred_B,
        labels=[
            0,
            1,
            2,
        ],
        normalize="true",
    )

    print(
        "STRICT:"
    )

    display(
        pd.DataFrame(
            cm_A,
            index=CLASSES,
            columns=CLASSES,
        ).round(3)
    )

    print(
        "TARGET-SPECIFIC NORMALIZATION:"
    )

    display(
        pd.DataFrame(
            cm_B,
            index=CLASSES,
            columns=CLASSES,
        ).round(3)
    )

    print(
        "\nTarget-normalized classification report:"
    )

    print(
        classification_report(
            y_all,
            pred_B,
            labels=[
                0,
                1,
                2,
            ],
            target_names=CLASSES,
            digits=4,
        )
    )

In [ ]:
# ============================================================
# CELL 15 — SAVE PROTOCOL
# ============================================================

protocol = {
    "model":
        "ATCNet-3C known-good reproduction",

    "dataset":
        "BCI-IV-2a",

    "input_shape":
        [22,640],

    "classes":
        CLASSES,

    "outer_evaluation":
        "9-subject LOSO",

    "source_validation":
        "stratified 80/20 trial split",

    "source_normalization":
        "channel-wise robust median/IQR",

    "seeds":
        [42,123],

    "learning_rate":
        9e-4,

    "batch_size":
        64,

    "target_A":
        "strict source normalization only",

    "target_B":
        "target-specific unlabeled median/IQR normalization",

    "target_labels_for_training":
        False,

    "target_labels_for_selection":
        False,
}

protocol_path = (
    RESULT_DIR
    / "atcnet_reproduction_ab_protocol.json"
)

with open(
    protocol_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        protocol,
        f,
        indent=2,
    )

print(
    "Protocol saved:",
    protocol_path
)

### Final reporting rule

Use **Experiment A** as the strict LOSO baseline.

Use **Experiment B** only as a separate unsupervised/transductive result because target EEG statistics are used for preprocessing.

The earlier known-good run recorded **80.56% standard ensemble and 82.87% TTA for S01**, while S02 and several other subjects demonstrated that source validation accuracy can be much higher than unseen-target accuracy. fileciteturn23file3L200-L212 fileciteturn23file1L83-L94

This notebook intentionally avoids DAFM, EEGCCT, DANN, MMD, CORAL, pseudo-labeling, and fusion so that the A/B target-normalization effect can be measured cleanly.